In [36]:
import os
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

Menggunakan device: cuda:0


### Inisialisasi Model, MTCNN, dan Transformasi

In [37]:
class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
IMG_SIZE = 300

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mtcnn = MTCNN(keep_all=False, select_largest=True, margin=60, post_process=False, device=device)

model = timm.create_model('efficientnet_b3', pretrained=False, num_classes=len(class_names))

model_path = '../Models/efficientnet_b3_model.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
# -------------------------

model = model.to(device)
model.eval() 
print(f"Model berhasil dimuat dari: {model_path}")

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_pat

Model berhasil dimuat dari: ../Models/efficientnet_b3_model.pth


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9184\1436280290.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=d

### Proses Inference 

In [38]:
import glob

test_dir = '../Data/test/' 

submission_df = pd.read_csv('../Data/samplesubmission.csv')
predictions = []

print("Memulai proses Prediksi Data Test...")
with torch.no_grad():
    for img_id in tqdm(submission_df['id']):
        search_pattern = os.path.join(test_dir, f"{img_id}.*")
        matching_files = glob.glob(search_pattern)
        
        if not matching_files:
            print(f"File tidak ditemukan untuk ID: {img_id} di folder {test_dir}")
            predictions.append('fake_unknown')
            continue
            
        img_path = matching_files[0] 
        
        try:
            img = Image.open(img_path).convert('RGB')
            face = mtcnn(img)
            
            if face is not None:
                face_img = transforms.ToPILImage()(face.byte())
                input_tensor = test_transform(face_img)
            else:
                input_tensor = test_transform(img)
                
            input_tensor = input_tensor.unsqueeze(0).to(device)
            
            outputs = model(input_tensor)
            _, predicted_idx = torch.max(outputs, 1)
            predicted_class = class_names[predicted_idx.item()]
            
            predictions.append(predicted_class)
            
        except Exception as e:
            print(f"Error memproses {img_path}: {e}")
            predictions.append('fake_unknown')

Memulai proses Prediksi Data Test...


  0%|          | 0/404 [00:00<?, ?it/s]

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [39]:
import os

model_name = 'efficientnet_b3'
submission_dir = '../Data/submission/'
os.makedirs(submission_dir, exist_ok=True)

submission_df['label'] = predictions
file_name = f'submission_{model_name}.csv'
submission_file_path = os.path.join(submission_dir, file_name)
submission_df.to_csv(submission_file_path, index=False)

print("="*40)
print(f"File submission disimpan")
print(f"Lokasi File: {submission_file_path}")
print("="*40)
display(submission_df.head(10))

File submission disimpan
Lokasi File: ../Data/submission/submission_efficientnet_b3.csv


,id,label
0,test_001,fake_printed
1,test_002,fake_screen
2,test_003,realperson
3,test_004,realperson
4,test_005,realperson
5,test_006,fake_mannequin
6,test_007,realperson
7,test_008,fake_mannequin
8,test_009,fake_mask
9,test_010,fake_screen
